In [ ]:


import yfinance as yf
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


ticker = "RELIANCE.NS" 
data = yf.download(ticker, start="2021-01-01", end="2026-01-01")

data['Returns'] = data['Close'].pct_change()


data['Vol_5d'] = data['Returns'].rolling(window=5).std()
data['Vol_20d'] = data['Returns'].rolling(window=20).std()


data['EMA_9'] = data['Close'].ewm(span=9, adjust=False).mean()
data['EMA_21'] = data['Close'].ewm(span=21, adjust=False).mean()
data['Signal_Line'] = data['EMA_9'] - data['EMA_21']

delta = data['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
data['RSI'] = 100 - (100 / (1 + rs))

data['Future_Return'] = data['Close'].shift(-5).pct_change(periods=5)
data['Target'] = np.where(data['Future_Return'] > 0.015, 1, 0)

data.dropna(inplace=True)

features = ['Vol_5d', 'Vol_20d', 'Signal_Line', 'RSI']
X = data[features]
y = data['Target']

split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))


[*********************100%***********************]  1 of 1 completed
C:\Users\SAKTHI NARAYANA RAO\AppData\Local\Temp\ipykernel_31792\2938277421.py:37: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['Future_Return'] = data['Close'].shift(-5).pct_change(periods=5)


              precision    recall  f1-score   support

           0       0.67      0.94      0.78       165
           1       0.29      0.05      0.09        79

    accuracy                           0.65       244
   macro avg       0.48      0.50      0.44       244
weighted avg       0.55      0.65      0.56       244

